# **Подготовка данных для обучения CatBoost модели, само обучение и применение**

**Проект:** Анализ и визуализация данных с использованием Yandex DataLens: исследование по прогнозированию CTR

**Автор:** Грицан М.А., студент группы БПМИ-247, 2 курса

**Дата:** 26-03-2026  

**Цель:** Обучение CatBoost модели, примение модели на тестовой выборке для получения результатов в kaggle соревновании ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/overview), а также сбор всех необходимых для дашборда метрик

#### Необходимые библиотеки и настройка графиков:

In [1]:
%pip install catboost -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import gc
import gzip
import zipfile
from collections import Counter
from datetime import datetime

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score

In [4]:
%config InlineBackend.figure_format = 'retina'

sns.set(style='darkgrid', palette='deep')

plt.rcParams['figure.figsize'] = 8, 5
plt.rcParams['font.size'] = 12
plt.rcParams['savefig.format'] = 'pdf'

### 0. Загрука набора данных c kaggle
Скачиваем все файлы с соревнования ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/data) для дальнейшего использования.

In [ ]:
os.environ['KAGGLE_API_TOKEN'] = "ВАШ_KAGGLE_API_TOKEN"

print("✅ Kaggle API ключ установлен!")

✅ Kaggle API ключ установлен!


In [5]:
%pip install kaggle -q
!kaggle competitions download -c avazu-ctr-prediction -p ../

Note: you may need to restart the kernel to use updated packages.
avazu-ctr-prediction.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
with zipfile.ZipFile('../avazu-ctr-prediction.zip', 'r') as zip_ref:
    zip_ref.extractall('../avazu-ctr-prediction')

### 1. Обучение CatBoost модели

При обучении будем использовать Out-Of-Time Validation, это лучше отражает качество моделей на времязависимых данных. Нам повезло и набор данных хранится уже осторированным. Как мы узнали ранее, в датасете $40 428 967$ строк. Оставим первые $30$ млн. из них на обучение, остальные олтложим для валидации. Получим, что примерно $74$% от тренировочного набора данных уйдет на обучающую выборку и $26$% на валидацонную.

Обучаться будем на GPU, а именно с помощью бесплатных GPU-часов от Kaggle.

Все признаки в датасете категориальные, кроме `hour`. Но этот признак нельзя передавать как численный: так мы неявно зададим порядок по времени. Я предлагаю вытащить из колонки 2 категориальных признака: номер часа в сутках и номер дня недели. Если оставить 2 признака числовыми, то мы опять сталкнемся с проблеммой неявного попрядка на данных.

#### Подготовим данные перед обучением модели

In [7]:
def data_tranformer(df: pd.DataFrame):
    dt = pd.to_datetime(df['hour'], format='%y%m%d%H')
    df['day_of_week'] = dt.dt.dayofweek
    df['hour_of_day'] = dt.dt.hour

    return df.drop(columns=['id', 'hour'], axis=1).astype(str)

In [ ]:
import pandas as pd
import gc
import os

os.makedirs(f'preprocessed_data', exist_ok=True)


print("⏳ Начинаем препроцессинг данных и разбивку на Train и Val...")

chunk_size = 5_000_000
train_file = '../avazu-ctr-prediction/train.gz'
chunk_iterator = pd.read_csv(train_file, compression='gzip', chunksize=chunk_size)

for chunk_num, chunk in enumerate(chunk_iterator, 1):
    processed_chunk = data_tranformer(chunk)

    if chunk_num <= 6:
        processed_chunk.to_csv(
            f'preprocessed_data/train_processed.csv',
            mode='w' if chunk_num == 1 else 'a',
            header=(chunk_num == 1),
            index=False
        )
        print(f"📦 Чанк №{chunk_num} обработан и добавлен в train_processed.csv")
    else:
        processed_chunk.to_csv(
            f'preprocessed_data/val_processed.csv',
            mode='w' if chunk_num == 7 else 'a',
            header=(chunk_num == 7),
            index=False
        )
        print(f"🧪 Чанк №{chunk_num} обработан и добавлен в val_processed.csv")

    del chunk, processed_chunk
    gc.collect()

print("✅ Препроцессинг завершен! Данные готовы к стримингу.")

⏳ Начинаем препроцессинг данных и разбивку на Train и Val...
📦 Чанк №1 обработан и добавлен в train_processed.csv
📦 Чанк №2 обработан и добавлен в train_processed.csv
📦 Чанк №3 обработан и добавлен в train_processed.csv
📦 Чанк №4 обработан и добавлен в train_processed.csv
📦 Чанк №5 обработан и добавлен в train_processed.csv
📦 Чанк №6 обработан и добавлен в train_processed.csv
🧪 Чанк №7 обработан и добавлен в val_processed.csv
🧪 Чанк №8 обработан и добавлен в val_processed.csv
🧪 Чанк №9 обработан и добавлен в val_processed.csv
✅ Препроцессинг завершен! Данные готовы к стримингу.


In [10]:
print("⏳ Создаем файл cd.txt...")

sample_df = pd.read_csv(f'preprocessed_data/train_processed.csv', nrows=0)
columns = sample_df.columns
target = 'click'

with open(f'preprocessed_data/cd.txt', 'w') as f:
    for i, col in enumerate(columns):
        if col == target:
            f.write(f"{i}\tLabel\n")
        else:
            f.write(f"{i}\tCateg\n")

print("✅ Файл cd.txt успешно создан!")

⏳ Создаем файл cd.txt...
✅ Файл cd.txt успешно создан!


### **Дальше уходим в Kaggle на бесплатную GPU. Это очень очень сильно ускорит обучение.**

Весь код ниже будет исполнен на ядре Kaggle, после чего итоговая модель будет скачана и сохранена локально.

# **Обучение CatBoost модели на GPU Kaggle**

### Необходимые импорты

In [ ]:
import gc

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool

In [ ]:
train_processed = '/kaggle/input/datasets/wwip62/preprocessed-avazu-ctr-prediction/train_processed.csv'
val_processed = '/kaggle/input/datasets/wwip62/preprocessed-avazu-ctr-prediction/val_processed.csv'
cd = '/kaggle/input/datasets/wwip62/preprocessed-avazu-ctr-prediction/cd.txt'

### Обучение

In [ ]:
print("⚙️ Инициализация Pool-объектов CatBoost'а...")

train_pool = Pool(
    data=train_processed,
    column_description=cd,
    has_header=True,
    delimiter=','
)

val_pool = Pool(
    data=val_processed,
    column_description=cd,
    has_header=True,
    delimiter=','
)

In [ ]:
model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=3,
    eval_metric='AUC',
    loss_function='Logloss',
    od_type='Iter',
    od_wait=200,
    use_best_model=True,
    max_ctr_complexity=2,
    ctr_leaf_count_limit=32,
    random_strength=0.5,
    has_time=True,
    bagging_temperature=1,
    border_count=64,
    random_seed=67,
    task_type='GPU',
    gpu_ram_part=0.7,
    verbose=100
)

print("🚀 Начинаем обучение...")
model.fit(train_pool, eval_set=val_pool)

print(f"\n🎉 Обучение завершено!")

Сохраним модель для дальнейшего использования локально.

In [ ]:
del train_pool, val_pool
gc.collect()

In [ ]:
model.save_model('catboost_ctr_model.cbm')

print(f"⚰️ Модель сохранена!")

На этом нужда в GPU все, закончилась.

### **Возвращаемся с обученной моделью**

Обучение модели на Kaggle прошло успешно, достанем ее и продолжим с ней работать.

In [ ]:
model_path = 'models/catboost_ctr_model.cbm'

print("⏳ Достаем обученную модель...")
model = CatBoostClassifier()
model.load_model(model_path)

print(f"Количество признаков в модели: {len(model.feature_names_)}")

Посмотрим на получившиеся на валидационной выборке метрики:

In [ ]:
val_pool = Pool(
    data='preprocessed_data/val_processed.csv',
    column_description='preprocessed_data/cd.txt',
    has_header=True,
    delimiter=','
) # здесь еще не прогоняли создание пула

In [ ]:
val_y_true = val_pool.get_label()

val_y_pred_proba = model.predict_proba(val_pool)[:, 1]
val_y_pred_class = model.predict(val_pool)

In [ ]:
roc_auc = roc_auc_score(val_y_true, val_y_pred_proba)
logloss = log_loss(val_y_true, val_y_pred_proba)
acc = accuracy_score(val_y_true, val_y_pred_class)

print(f"📊 Итоговые метрики:")

metrics_df = pd.DataFrame({
    'Метрика': ['ROC-AUC Score', 'Log Loss', 'Accuracy'],
    'Значение': [roc_auc, logloss, acc]
})
metrics_df['Значение'] = metrics_df['Значение'].round(4)

metrics_df

### 2. Применение модели на тестовой выборке и отправка решения на kaggle

Считаем тестовую выборку:

In [ ]:
warnings.filterwarnings('ignore')

test_file = "../avazu-ctr-prediction/test.gz"

print("⏳ Читаем test.gz...")
test_df = pd.read_csv(test_file, compression='gzip', dtype={'id': str})

print(f"✅ Итого загружено: {f"{len(test_df):_}".replace('_', ' ')} строк")

ram_usage = test_df.memory_usage(deep=True).sum() / 1024**3
print(f"📊 Объем памяти DataFrame: {ram_usage:.2f} GB")

Применим модель к тестовому набору данных:

In [ ]:
ids = test_df['id']
X_test = data_tranformer(test_df)

del test_df
gc.collect()

print("\n⏳ Генерация предсказаний...")
y_pred_proba = model.predict_proba(X_test)[:, 1]

del X_test
gc.collect()

Выведем минимальную статистику по полученным предсказаниям:

In [ ]:
stats_data = [
    ('Средняя вероятность', y_pred_proba.mean()),
    ('Медианная вероятность', np.median(y_pred_proba)),
    ('Стандартное отклонение', y_pred_proba.std()),
    ('Минимум', y_pred_proba.min()),
    ('Максимум', y_pred_proba.max())
]
stats_df = pd.DataFrame(stats_data, columns=['Статистика', 'Значение'])
stats_df['Значение'] = stats_df['Значение'].round(4)

print("📊 Статистики предсказаний на тестовой выборке:")
display(stats_df)


quantiles = []
for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
    quantiles.append((f'{int(q*100)}%', np.quantile(y_pred_proba, q)))
quantiles_df = pd.DataFrame(quantiles, columns=['Квантиль', 'Значение'])

print("📊 Квантили на тестовой выборке:")
display(quantiles_df)


*Наконец, сохраненим результат и сделаем посылку submission'а на kaggle:*

In [ ]:
submission_file = 'data/submission.csv'
os.makedirs('data', exist_ok=True)

In [ ]:
submission = pd.DataFrame({'id': ids, 'click': y_pred_proba})
submission.to_csv(submission_file, index=False)
print(f"✅ Результаты успешно сохранены в: {submission_file}")

In [ ]:
!kaggle competitions submit -c avazu-ctr-prediction -f data/submission.csv  -m "CatBoost model with more iterations and params, learning by Kaggle GPU"